# Generalization and Regularization


Figures are from Prince, *[Understanding Deep Learning](https://udlbook.github.io/udlbook/)*
(UDL), unless another source is cited. This lecture covers UDL chapters 8 and 9.
Code cells adapted from the UDL notebooks (MIT license,
[github.com/udlbook/udlbook/tree/main/Notebooks](https://github.com/udlbook/udlbook/tree/main/Notebooks)).
If the train/validation/test split or the pictures of over- and underfitting are
new to you, watch the
[A4 Measuring Performance refresher](A4-Measuring-Performance-Refresher.qmd) first.


## Generalization and regularization

- MNIST-1D: a small dataset that tells models apart
- Noise, bias and variance: decomposing the test error
- Reducing variance, reducing bias, and the trade-off between them
- Double descent
- Curse of dimensionality and the weird geometry of high-dimensional space
- Choosing hyperparameters
- Regularization: explicit (L2, weight decay, AdamW) and implicit (GD, SGD)
- Early stopping, ensembles, dropout, noise
- Transfer learning, multi-task and self-supervised learning, data augmentation

# MNIST-1D: a dataset that tells models apart

To measure how well a trained model generalizes, we need a dataset that is small
enough to experiment with quickly but hard enough to tell models apart. MNIST-1D is
that dataset, and we use it throughout this lecture.


## MNIST-1D: scaling down deep learning

> "A large number of deep learning innovations including
> [dropout](https://jmlr.org/papers/v15/srivastava14a.html),
> [Adam](https://arxiv.org/abs/1412.6980),
> [convolutional networks](http://yann.lecun.com/exdb/publis/pdf/lecun-89e.pdf),
> [generative adversarial networks](https://arxiv.org/abs/1406.2661), and
> [variational autoencoders](https://arxiv.org/abs/1312.6114) began life as MNIST
> experiments. Once these innovations proved themselves on small-scale experiments,
> scientists found ways to scale them to larger and more impactful applications."

S. Greydanus, "Scaling down Deep Learning." arXiv, Nov. 2020. doi:
[10.48550/arXiv.2011.14439](https://doi.org/10.48550/arXiv.2011.14439).
<https://github.com/greydanus/mnist1d>


![](figs/06-Generalization-and-Regularization/mnist-digits.png)

MNIST (28x28 grayscale, 60K train / 10K test) "is to deep learning what fruit
flies are to genetics research", but it poorly differentiates model performance:

| Model Type          | Accuracy |
|---------------------|----------|
| Logistic Regression | 94%      |
| MLP                 | 99+%     |
| CNN                 | 99+%     |


## MNIST-1D construction

Fixed, 1-D, length-12 templates for each of 10 digit classes.

Generate the dataset by programmatically applying 6 parametric transformations:
pad, shear, translate, correlated noise, i.i.d. noise, interpolation.

See the [building-mnist1d notebook](https://github.com/greydanus/mnist1d/blob/master/notebooks/building-mnist1d.ipynb)
([Colab](https://colab.research.google.com/github/greydanus/mnist1d/blob/master/notebooks/building-mnist1d.ipynb)).


![](figs/06-Generalization-and-Regularization/mnist1d-construction.png)



## MNIST-1D differentiates model types

Differentiates performance of different model types much more than MNIST.

**Shuffle**: dataset was permuted along the spatial dimension. This 'shuffled'
version measured each of the models' performances in the absence of local spatial
structure.


![](figs/06-Generalization-and-Regularization/mnist1d-test-accuracy.png)


| Dataset              | Logistic regression | Fully connected model | Convolutional model | GRU model    | Human expert      |
|----------------------|---------------------|-----------------------|---------------------|--------------|-------------------|
| MNIST                | $94 \pm 0.5$        | $> 99$                | $> 99$              | $> 99$       | $> 99$            |
| MNIST-1D             | $32 \pm 1$          | $68 \pm 2$            | $94 \pm 2$          | $91 \pm 2$   | $96 \pm 1$        |
| MNIST-1D (shuffled)  | $32 \pm 1$          | $68 \pm 2$            | $56 \pm 2$          | $57 \pm 2$   | $\approx 30 \pm 10$ |

## Visualizing MNIST and MNIST-1D with tSNE

![Visualizing the MNIST and MNIST-1D datasets with tSNE. The well-defined clusters in the MNIST plot indicate that the majority of the examples are separable via a kNN classifier in pixel space. The MNIST-1D plot, meanwhile, reveals a lack of well-defined clusters which suggests that learning a nonlinear representation of the data is much more important to achieve successful classification. Thanks to Dmitry Kobak for making this plot. ([\@hippopedoid](https://twitter.com/hippopedoid))](figs/06-Generalization-and-Regularization/mnist-tsne.png)


## Dataset and network

**Dataset**: 1-D, length 40 samples; 4,000 training and 1,000 test samples (80/20 split)

![](figs/06-Generalization-and-Regularization/mnist1d-samples.png)

**Network**: 40 inputs, 10 outputs, two hidden layers of 100 units (15,210 parameters).
SGD with batch size 100, learning rate 0.05, momentum 0.9, halved every 10 epochs (UDL notebook 8.1). Say 50 epochs, how many steps total?


```python
model = torch.nn.Sequential(
    torch.nn.Linear(40, 100),
    torch.nn.ReLU(),
    torch.nn.Linear(100, 100),
    torch.nn.ReLU(),
    torch.nn.Linear(100, 10))

# choose cross entropy loss function
loss_function = torch.nn.CrossEntropyLoss()
# construct SGD optimizer and initialize learning rate and momentum
optimizer = torch.optim.SGD(model.parameters(), lr = 0.05, momentum = 0.9)
# object that decreases learning rate by half every 10 epochs
scheduler = StepLR(optimizer, step_size=10, gamma=0.5)
# load the data into a class that creates the batches
data_loader = DataLoader(TensorDataset(x_train,y_train), batch_size=100, shuffle=True)
...
# inference – just choose the max
pred_test = model(x_test)
_, predicted_test_class = torch.max(pred_test.data, 1)
```



## The generalization gap

![](figs/06-Generalization-and-Regularization/mnist1d-train-test-curves.png)

Training error and loss both fall to zero; test error levels off near 40% and the
test loss turns back up. The model has not **generalized** well to the new data.
Try it yourself in [UDL Notebook 8.1 MNIST_1D_Performance](https://github.com/udlbook/udlbook/blob/main/Notebooks/Chap08/8_1_MNIST_1D_Performance.ipynb).

# Noise, bias, and variance

The gap between training and test performance has to come from somewhere. Using a
simple regression toy model, we split the test error into three sources: noise, bias
and variance. The derivation is short, and it is the kind of thing a quiz can ask
you to reproduce.


## Regression example with a toy model

**Dataset**

![](figs/06-Generalization-and-Regularization/toy-dataset.png)

"True" function $y = e^{\sin(2\pi x)}$, with $x_i \sim \mathcal{U}(i/n, (i+1)/n)$
(one point per grid cell) and Gaussian noise on $y$, $\mathcal{N}(0, \sigma_y^2)$
with $\sigma_y = 0.3$.


**Model**

![](figs/06-Generalization-and-Regularization/toy-model.png)

- $D$ hidden units; first layer fixed so the "joints" divide the interval evenly,
  e.g. $0, 1/D, 2/D, \ldots, (D-1)/D$; second layer trained
- Linear in $\mathbf{h}$, so convex cost and a closed-form best solution
- **A piecewise linear model with $D$ regions.**


## Noise, bias, and variance: noise

![](figs/06-Generalization-and-Regularization/noise.png)


- Genuine stochastic nature of the underlying model
- Noise in measurements, e.g. from sensors
- Some variables not observed
- Data mislabeled


## Where measurement noise comes from

| Noise Type        | Domain | Source                  | Distribution         | Signal Dependent?        |
|-------------------|--------|-------------------------|----------------------|--------------------------|
| **Photon Shot**   | Image  | Quantum nature of light | Poisson              | Yes                      |
| **Dark Current**  | Image  | Heat in Silicon         | Poisson/Gaussian     | No (Temp/Time dependent) |
| **Read Noise**    | Image  | Electronics/Amplifiers  | Gaussian             | No                       |
| **FPN**           | Image  | Manufacturing defects   | Constant (Spatial)   | No                       |
| **Thermal**       | Audio  | Electron motion         | Gaussian (White)     | No                       |
| **1/f (Flicker)** | Audio  | Material impurities     | Pink Noise ($1/f$)   | No                       |

- Real sensor noise is not one Gaussian: shot noise scales with the signal, fixed
  pattern noise is the same in every frame (a model *can* overfit to it), and
  quantization error is deterministic.
- The slide-by-slide tour of image and audio sensor noise is in the
  [A4 refresher](A4-Measuring-Performance-Refresher.qmd).

## Noise, bias, and variance: bias and variance


![](figs/06-Generalization-and-Regularization/noise-bias-variance.png)


**Bias** occurs because the model lacks precision or capacity to accurately match
the underlying function.

E.g. optimal fit with 3 hidden units and 3 line segments


In practice, we have limited noisy training data (orange points)

**Variance**: variability every time we capture training data and also from
stochastic learning algorithms.


## Mathematical formulation of test error

*UDL 8.2.2*

- 1-D regression where the underlying data generation process (unobservable) has
  additive noise with variance $\sigma^2$.


![](figs/06-Generalization-and-Regularization/data-generation-sample.png)


- For each $x$ there is a distribution of $y[x]$ which is $Pr(y|x)$, with
  **expected value** (mean) $\mu[x]$ and noise variance $\sigma^2$:

$$
\mu[x] = \mathbb{E}_y\big[y[x]\big] = \int y[x] Pr(y|x) dy, \qquad
\sigma^2 = \mathbb{E}_y\Big[\big(\mu[x] - y[x]\big)^2\Big]
$$


**Note**

The name is a bit nuanced and maybe a little confusing: $\mu[x]$ is the expected
value of $y$ for a given $x$. Maybe $\mu_y[x]$ would be better??



## Loss at $x$: subtract and add $\mu[x]$

*UDL 8.2.2*

- The loss at input $x$, $L[x]$, between the model prediction
  $\mathrm{f}[x, \boldsymbol{\phi}]$ and the output at $x$, $y[x]$, is
  $L[x] = \big(\mathrm{f}[x, \boldsymbol{\phi}] - y[x]\big)^2$.
- **Mathematical "trick"**: subtract and add $\mu[x]$, group, then multiply out

$$
\begin{aligned}
L[x] &= \big(\mathrm{f}[x, \boldsymbol{\phi}] - y[x]\big)^2 \\
&= \Big(\big(\mathrm{f}[x, \boldsymbol{\phi}] - \mu[x]\big) + \big(\mu[x] - y[x]\big)\Big)^2 \\
&= \big(\mathrm{f}[x, \boldsymbol{\phi}] - \mu[x]\big)^2 + 2\big(\mathrm{f}[x, \boldsymbol{\phi}] - \mu[x]\big)\big(\mu[x] - y[x]\big) + \big(\mu[x] - y[x]\big)^2,
\end{aligned}
$$

## Take the expectation over $y$

- We are treating $y[x]$ as a random variable, so we can take the expectation of
  $L[x]$ with respect to $y$. Using the linear properties of expectation, we move it
  into the sum:

$$
\begin{aligned}
\mathbb{E}_y\big[L[x]\big] &= \mathbb{E}_y\Big[\big(\mathrm{f}[x, \boldsymbol{\phi}] - \mu[x]\big)^2 + 2\big(\mathrm{f}[x, \boldsymbol{\phi}] - \mu[x]\big)\big(\mu[x] - y[x]\big) + \big(\mu[x] - y[x]\big)^2\Big] \\
&= \big(\mathrm{f}[x, \boldsymbol{\phi}] - \mu[x]\big)^2 + 2\big(\mathrm{f}[x, \boldsymbol{\phi}] - \mu[x]\big)\big(\mu[x] - \mathbb{E}_y\big[y[x]\big]\big) + \mathbb{E}_y\big[(\mu[x] - y[x])^2\big]
\end{aligned}
$$

$$
\begin{aligned}
\phantom{\mathbb{E}_y\big[L[x]\big]} &= \big(\mathrm{f}[x, \boldsymbol{\phi}] - \mu[x]\big)^2 + 2\big(\mathrm{f}[x, \boldsymbol{\phi}] - \mu[x]\big) \cdot 0 + \mathbb{E}_y\Big[\big(\mu[x] - y[x]\big)^2\Big]
\end{aligned}
$$

Middle term becomes zero because $\mathbb{E}_y[y[x]] = \mu[x]$.


To get two terms…

$$
\begin{aligned}
\phantom{\mathbb{E}_y\big[L[x]\big]} &= \underbrace{\big(\mathrm{f}[x, \boldsymbol{\phi}] - \mu[x]\big)^2}_{\text{squared deviation of model from true mean}} + \underbrace{\sigma^2}_{\text{noise variance}}, \qquad (8.3)
\end{aligned}
$$


## Expectation w.r.t. the training dataset

- The first term can be further split into **bias** and **variance**.
- Parameters $\boldsymbol{\phi}$ of the model $\mathrm{f}[x, \boldsymbol{\phi}]$
  depend on the training dataset $\mathcal{D} = \{x_i, y_i\}$, e.g.
  $\mathrm{f}[x, \boldsymbol{\phi}[\mathcal{D}]]$, and the training dataset is a noisy
  sample of the true data.
- So we can write the expected model output $\mathrm{f}_{\mu}[x]$ w.r.t. all
  possible datasets $\mathcal{D}$

$$
\mathrm{f}_{\mu}[x] = \mathbb{E}_{\mathcal{D}}\Big[\mathrm{f}\big[x, \boldsymbol{\phi}[\mathcal{D}]\big]\Big]
$$

- Same trick: subtract and add $\mathrm{f}_{\mu}[x]$ in the first term and multiply out

$$
\begin{aligned}
&\big(\mathrm{f}[x, \boldsymbol{\phi}[\mathcal{D}]] - \mu[x]\big)^2 \\
&\quad = \Big(\big(\mathrm{f}[x, \boldsymbol{\phi}[\mathcal{D}]] - \mathrm{f}_{\mu}[x]\big) + \big(\mathrm{f}_{\mu}[x] - \mu[x]\big)\Big)^2 \\
&\quad = \big(\mathrm{f}[x, \boldsymbol{\phi}[\mathcal{D}]] - \mathrm{f}_{\mu}[x]\big)^2 + 2\big(\mathrm{f}[x, \boldsymbol{\phi}[\mathcal{D}]] - \mathrm{f}_{\mu}[x]\big)\big(\mathrm{f}_{\mu}[x] - \mu[x]\big) + \big(\mathrm{f}_{\mu}[x] - \mu[x]\big)^2.
\end{aligned}
$$


## Mathematical formulation of test error (result)

- Then take the expectation w.r.t. training dataset $\mathcal{D}$:

$$
\mathbb{E}_{\mathcal{D}}\Big[\big(\mathrm{f}[x, \boldsymbol{\phi}[\mathcal{D}]] - \mu[x]\big)^2\Big] = \mathbb{E}_{\mathcal{D}}\Big[\big(\mathrm{f}[x, \boldsymbol{\phi}[\mathcal{D}]] - \mathrm{f}_{\mu}[x]\big)^2\Big] + \big(\mathrm{f}_{\mu}[x] - \mu[x]\big)^2,
$$

Middle term on previous slide goes to zero. Check!

- We can then write $\mathbb{E}_{\mathcal{D}}\Big[\mathbb{E}_y\big[L[x]\big]\Big]$

$$
\mathbb{E}_{\mathcal{D}}\Big[\mathbb{E}_y\big[L[x]\big]\Big] = \underbrace{\mathbb{E}_{\mathcal{D}}\Big[\big(\mathrm{f}[x, \boldsymbol{\phi}[\mathcal{D}]] - \mathrm{f}_{\mu}[x]\big)^2\Big]}_{\text{variance}} + \underbrace{\big(\mathrm{f}_{\mu}[x] - \mu[x]\big)^2}_{\text{bias}} + \underbrace{\sigma^2}_{\text{noise}}.
$$


## Summary for regression with least squares error

$$
\mathbb{E}_{\mathcal{D}}\Big[\mathbb{E}_y\big[L[x]\big]\Big] = \underbrace{\mathbb{E}_{\mathcal{D}}\Big[\big(\mathrm{f}[x, \boldsymbol{\phi}[\mathcal{D}]] - f_{\mu}[x]\big)^2\Big]}_{\text{variance}} + \underbrace{\big(f_{\mu}[x] - \mu[x]\big)^2}_{\text{bias}} + \underbrace{\sigma^2}_{\text{noise}}
$$

- $\mathbb{E}_{\mathcal{D}}$: expectation over noise in training data
- $\mathbb{E}_y$: expectation over noise in test data
- $\mathrm{f}[x, \boldsymbol{\phi}[\mathcal{D}]]$: actual model
- $f_{\mu}[x]$: the model averaged over all training sets of this size, $\mathbb{E}_{\mathcal{D}}\big[\mathrm{f}[x, \boldsymbol{\phi}[\mathcal{D}]]\big]$
- $\mu[x]$: true function


**Warning**

The interactions between variance, bias and noise are more complex for other loss
functions, e.g. cross entropy loss in classification. Adding variance might
counteract bias and improve classification results by pushing samples across
classification boundaries…



The decomposition, checked by simulation on the toy problem: draw many training
sets of 15 noisy points, fit the $D = 3$ piecewise linear model to each in
closed form, and measure the three terms averaged over $x \in [0, 1]$. Then
draw fresh noisy test labels and measure the expected test error directly. The
two numbers agree up to Monte Carlo error. Change `D`, `n_data` or `sigma_y`
and rerun.

In [ ]:
# Adapted from UDL notebook 8.2 (MIT)
import numpy as np
import matplotlib.pyplot as plt

sigma_y = 0.3       # noise standard deviation
n_data = 15         # training points per dataset
n_datasets = 2000   # datasets drawn to estimate E_D
D = 3               # hidden units (linear regions)

def true_function(x):
    return np.exp(np.sin(2 * np.pi * x))

def generate_data(n, rng, sigma_y=sigma_y):
    x = (np.arange(n) + rng.uniform(size=n)) / n      # one point per grid cell
    return x, true_function(x) + rng.normal(0, sigma_y, n)

def features(x, D):
    """Bias column plus D ReLU hidden units with fixed joints at 0, 1/D, ..., (D-1)/D."""
    joints = np.arange(D) / D
    return np.column_stack([np.ones_like(x), np.maximum(x[:, None] - joints, 0)])

def fit(x, y, D):
    """Closed-form least squares for the trained second layer (beta, omega)."""
    return np.linalg.lstsq(features(x, D), y, rcond=None)[0]

def fits_over_datasets(D, n_data, n_datasets, rng):
    """Each row: the model fitted to one training set, evaluated on x_grid."""
    A_grid = features(x_grid, D)
    return np.array([A_grid @ fit(*generate_data(n_data, rng), D)
                     for _ in range(n_datasets)])

rng = np.random.default_rng(0)
x_grid = np.linspace(0, 1, 201)
mu = true_function(x_grid)                          # mu[x], the true mean

F = fits_over_datasets(D, n_data, n_datasets, rng)  # f[x, phi[D]]
f_mu = F.mean(axis=0)                               # f_mu[x] = E_D[f]
variance = np.mean((F - f_mu) ** 2)
bias = np.mean((f_mu - mu) ** 2)
noise = sigma_y ** 2

y_test = mu + rng.normal(0, sigma_y, F.shape)       # fresh test labels y[x]
test_error = np.mean((F - y_test) ** 2)

print(f"variance {variance:.4f} + bias {bias:.4f} + noise {noise:.4f} "
      f"= {variance + bias + noise:.4f}")
print(f"expected test error measured directly:    {test_error:.4f}")

# Reducing variance, reducing bias, and the trade-off

The decomposition says variance comes from fitting a model to one particular noisy
sample of data, and bias from a model that lacks the capacity to match the true
function. The toy model shows how each shrinks, and why fixing one aggravates the
other.


## Variance shrinks with more data


When measuring (capturing) 6 different data samples with a fixed model (e.g. 3
hidden units), we get different optimal fits every time. Can reduce variance by
adding more samples.

![](figs/06-Generalization-and-Regularization/variance-100-samples.png)


## Reducing bias $\rightarrow$ increases variance!!


![](figs/06-Generalization-and-Regularization/bias-variance-regions.png)


- We can reduce bias by adding more model capacity, in this case more hidden units
  (top row: 3, 5, 10 regions fit the true function ever more closely).

- But with a fixed sample of 15 noisy points, the 10-region model describes the
  training data better, not the true underlying function: there are many ways to
  fit 15 points, so the fits vary more from sample to sample (bottom row).


## Bias and variance trade-off for the simple linear model

![](figs/06-Generalization-and-Regularization/bias-variance-tradeoff.png)

Model capacity: number of hidden units


![](figs/06-Generalization-and-Regularization/toy-model-network.png)


$$
\mathbb{E}_{\mathcal{D}}\Big[\mathbb{E}_y\big[L[x]\big]\Big] = \underbrace{\mathbb{E}_{\mathcal{D}}\Big[\big(\mathrm{f}[x, \boldsymbol{\phi}[\mathcal{D}]] - f_{\mu}[x]\big)^2\Big]}_{\text{variance}} + \underbrace{\big(f_{\mu}[x] - \mu[x]\big)^2}_{\text{bias}} + \underbrace{\sigma^2}_{\text{noise}}
$$

But does picking model capacity to minimize bias & variance hold for more complex
data and models? Back to MNIST-1D to find out.


The trade-off curve from the simulation above: sweep the number of hidden units
with 15 training points per dataset. The noise term, $\sigma_y^2 = 0.09$, adds
the same constant at every capacity, so it is left off. Try `n_data = 50` and
watch the minimum move right: more data buys room for more capacity.

In [ ]:
#| code-fold: true
#| fig-cap: "Bias, variance and their sum against model capacity for the toy problem, 15 training points per dataset and 2,000 datasets per capacity."
#| fig-alt: "Line plot of mean squared error against hidden units from 1 to 12. Bias (orange) is about 0.30 for 1 and 2 units, drops to 0.06 at 3 and near 0 from 4 on. Variance (blue) rises slowly from about 0.015 to 0.1 at 10 units, then sharply to about 0.22 at 12. Their sum (dashed) is lowest at 4 hidden units."
# Adapted from UDL notebook 8.2 (MIT)
n_data = 15
capacities = np.arange(1, 13)
rng = np.random.default_rng(1)
bias_D, variance_D = [], []
for D_k in capacities:
    F = fits_over_datasets(D_k, n_data, n_datasets, rng)
    f_mu = F.mean(axis=0)
    variance_D.append(np.mean((F - f_mu) ** 2))
    bias_D.append(np.mean((f_mu - mu) ** 2))
bias_D, variance_D = np.array(bias_D), np.array(variance_D)
best = capacities[np.argmin(bias_D + variance_D)]
print(f"bias + variance is smallest at D = {best} hidden units")

fig, ax = plt.subplots(figsize=(6, 3.6))
ax.plot(capacities, bias_D, "o-", color="#eb6834", label="bias")
ax.plot(capacities, variance_D, "o-", color="#2a78d6", label="variance")
ax.plot(capacities, bias_D + variance_D, "k--", label="bias + variance")
ax.set_xlabel("Model capacity (hidden units $D$)")
ax.set_ylabel("Mean squared error")
ax.set_ylim(0, 0.5)
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

# Double descent

On MNIST-1D the classical trade-off does not appear as expected: test error keeps
falling as capacity grows, and adding label noise reveals a second descent.


## Train and test error versus hidden layer size

- 10,000 training examples
- 5,000 test examples
- Two hidden layers
- Adam optimizer
- Step size of 0.005
- Full batch
- 4000 training steps

**Test error keeps decreasing even as we keep increasing model capacity!**

Dashed line: training parameters = training examples

Model has *memorized* the training set. **Why do we say that?**


![](figs/06-Generalization-and-Regularization/mnist1d-no-label-noise.png)


## Now randomize 15% of the training labels


![](figs/06-Generalization-and-Regularization/double-descent-regimes.png)


Now we see what looks like the bias-variance trade-off as we increase capacity to the
point where the model fits the training data. But then??? Test error descends
*again*: **double descent**, with a classical (under-parameterized) regime, a
critical regime at the interpolation threshold, and a modern (over-parameterized)
regime.

**Note**

Reminder: vertical dashed line is where: # training parameters = # training samples


## Same phenomenon shows up on MNIST and CIFAR100

![](figs/06-Generalization-and-Regularization/double-descent-mnist-cifar.png)


**Note**

Reminder: vertical dashed line is where: # training parameters = # training samples


## Double descent: it must be happening between the data points

- Note that training loss is very close to zero.
- Whatever is happening isn't happening at training data points
- Model never sees test set during training
- Must be happening between the data points??


![](figs/06-Generalization-and-Regularization/double-descent-train-zero.png)


## Smoother functions with more hidden units


![](figs/06-Generalization-and-Regularization/smooth-interpolation-hidden-units.png)

![](figs/06-Generalization-and-Regularization/zero-loss-solutions.png)


Potential explanation:

- can make smoother functions with more hidden units
- being smooth between the datapoints is a reasonable thing to do

But why?

All three fits in the second figure pass through the six points with zero loss.
They are equivalent in terms of loss, so **why should the model choose the smooth
solution?**


# Curse of dimensionality & weird properties of high dimensional space

Why should the space between data points matter so much? In high dimensions, almost
all of the space is between the data points.


## Curse of dimensionality

- 40-dimensional data
- 10,000 data points
- Consider quantizing each dimension into 10 bins
- $10^{40}$ bins
- 1 data point per $10^{36}$ bins
- *So most of the space has to be interpolated*
- The tendency of high-dimensional space to overwhelm the number of data points is
  called the **curse of dimensionality**


![2D: 10x10=100 bins](figs/06-Generalization-and-Regularization/grid-2d.png)

![3D: 10x10x10=1000 bins](figs/06-Generalization-and-Regularization/grid-3d.png)


## An aside on the curse: distances collapse

Generate 1,000 normally distributed samples in:

- 2D
- 3D
- 100D
- 1000D

Calculate the ratio of distances between the farthest and closest points.

**Approaches 1!!**


![](figs/06-Generalization-and-Regularization/normal-samples-2d.png)
![](figs/06-Generalization-and-Regularization/normal-samples-3d.png)

![](figs/06-Generalization-and-Regularization/distance-ratio.png)


## An aside on the curse: volumes of hyperspheres

![](figs/06-Generalization-and-Regularization/hypersphere-volume-unit-radius.png)

![](figs/06-Generalization-and-Regularization/orange-peel.png)


![Volume of unit diameter hypersphere, also the volume ratio to a unit hypercube.](figs/06-Generalization-and-Regularization/hypersphere-volume-unit-diameter.png)

![The proportion of volume within 1% of the surface (the outer 1% of the radius; the figure's title says diameter).](figs/06-Generalization-and-Regularization/hypersphere-outer-shell.png)


![Unit diameter hypersphere in a unit hypercube.](figs/06-Generalization-and-Regularization/sphere-in-cube.png)

"All the volume goes to the peel of the orange, not the pulp."

See also ["An Adventure in the Nth Dimension", American Scientist](https://www.americanscientist.org/article/an-adventure-in-the-nth-dimension)


Both asides in code. First, 1,000 standard normal points in each dimension:
the ratio of the largest to the smallest pairwise distance, and the spread of
all pairwise distances relative to their mean. The smallest distance depends on
which two points happen to land close together, so the low-dimensional ratios
change a lot with the seed; the collapse toward 1 does not. Second, the volume
of a $d$-dimensional ball of radius $r$,
$V_d(r) = \pi^{d/2} r^d / \Gamma(d/2 + 1)$, and the fraction of it in the outer
shell between $0.99r$ and $r$, which is $1 - 0.99^d$.

In [ ]:
# Adapted from UDL notebook 8.4 (MIT)
from math import gamma, pi

rng = np.random.default_rng(0)
n_points = 1000
iu = np.triu_indices(n_points, k=1)                 # each pair once, no self-distances
for d in [2, 3, 100, 1000]:
    X = rng.standard_normal((n_points, d))
    sq = np.sum(X ** 2, axis=1)
    dist = np.sqrt(np.maximum(sq[:, None] + sq[None, :] - 2 * X @ X.T, 0))[iu]
    print(f"d = {d:4d}: largest / smallest distance = {dist.max() / dist.min():8.2f}, "
          f"std / mean of distances = {dist.std() / dist.mean():.3f}")

def ball_volume(d, r):
    return pi ** (d / 2) * r ** d / gamma(d / 2 + 1)

print()
v_unit_radius = {d: ball_volume(d, 1.0) for d in range(1, 21)}
d_peak = max(v_unit_radius, key=v_unit_radius.get)
print(f"unit radius:   volume peaks at d = {d_peak} ({v_unit_radius[d_peak]:.2f}), "
      f"d = 20: {v_unit_radius[20]:.3f}")
print("unit diameter: " + ", ".join(f"d = {d}: {ball_volume(d, 0.5):.3f}" for d in [1, 2, 3, 8, 20]))

print()
for d in [1, 2, 10, 20, 50, 100, 150, 200, 250, 300]:
    shell = 1 - 0.99 ** d          # (V(r) - V(0.99 r)) / V(r)
    print(f"d = {d:3d}: fraction of volume in outer 1% of radius = {shell:.3f}")

## Smooth interpolation between data points

Returning to the fits through six points with more and more hidden units, the
potential explanations for why a well-trained model interpolates smoothly:

- Network initialization may encourage smoothness and the model doesn't depart much
  during training
- Through **implicit** and **explicit regularization** (the second half of this
  lecture), the (well trained) model tends to interpolate smoothly between training
  data points.

# Choosing hyperparameters

In practice we cannot compute bias or variance, so we need an empirical way to pick
capacity and the other choices that shape training.


## Choosing hyperparameters

- Don't know bias or variance
- Don't know how much capacity to add
- How do we choose capacity in practice?
  - Or model structure
  - Or training algorithm
  - Or learning rate
- Third data set – **validation set**
  - Train models with different hyperparameters on training set
  - Choose best hyperparameters with validation set
  - Test once with test set

# Regularization

The first half of the lecture ended with a puzzle: a model with zero training loss
has many functions to choose from between the data points, and something steers it
toward the smooth ones. Regularization is the collective name for everything that
does the steering, on purpose or by accident.


## What is regularization?

- Why is there a generalization gap between training and test data?
  - Overfitting (model describes statistical peculiarities)
  - Model unconstrained in areas where there are no training examples
- **Regularization** = methods to reduce the generalization gap
- Technically means adding terms to loss function
- But colloquially means any method (hack) to reduce gap between training and test data

- Explicit regularization
- Implicit regularization
- Early stopping, ensembling, dropout, adding noise
- Transfer learning, multi-task learning, self-supervised learning
- Data augmentation


# Explicit regularization

The most direct form of regularization changes the objective itself: we add a
term to the loss function that is smaller for the parameters we prefer.


## Explicit regularization

- Standard loss function:

$$
\begin{aligned}
\hat{\boldsymbol{\phi}} &= \underset{\boldsymbol{\phi}}{\mathrm{argmin}}\bigl[\mathrm{L}[\boldsymbol{\phi}]\bigr] \\
&= \underset{\boldsymbol{\phi}}{\mathrm{argmin}}\left[\sum_{i=1}^{I} \ell_i[\mathbf{x}_i, \mathbf{y}_i]\right]
\end{aligned}
$$

- Regularization adds an extra term

$$
\hat{\boldsymbol{\phi}} = \underset{\boldsymbol{\phi}}{\mathrm{argmin}}\left[\sum_{i=1}^{I} \ell_i[\mathbf{x}_i, \mathbf{y}_i] + \lambda \cdot \mathrm{g}[\boldsymbol{\phi}]\right]
$$


- Where $\mathrm{g}[\boldsymbol{\phi}]$ is smaller for preferred parameters
- $\lambda > 0$ controls the strength of influence


## Explicit regularization: an example

![Loss function for the Gabor model (Lecture 04). Cyan points denote local minima.](figs/06-Generalization-and-Regularization/gabor-loss.png)


![Example of a regularization function that prefers parameters close to 0.](figs/06-Generalization-and-Regularization/gabor-regularization.png)


![Fewer local minima and the absolute minimum has moved.](figs/06-Generalization-and-Regularization/gabor-loss-plus-regularization.png)


## Probabilistic interpretation: regularizer = prior

- Maximum likelihood:

$$
\hat{\boldsymbol{\phi}} = \underset{\boldsymbol{\phi}}{\mathrm{argmax}}\left[\prod_{i=1}^{I} Pr(\mathbf{y}_i \mid \mathbf{x}_i, \boldsymbol{\phi})\right]
$$

- Regularization is equivalent to adding a **prior** over parameters: what you know
  about the parameters *before* seeing the data. This is the *maximum a posteriori*
  or **MAP** criterion:

$$
\hat{\boldsymbol{\phi}} = \underset{\boldsymbol{\phi}}{\mathrm{argmax}}\left[\prod_{i=1}^{I} Pr(\mathbf{y}_i \mid \mathbf{x}_i, \boldsymbol{\phi}) Pr(\boldsymbol{\phi})\right]
$$

- Converting to negative log likelihood (take $-\log[\cdot]$ and minimize), the
  product becomes the usual sum of per-example losses and the prior becomes the
  regularizer:

$$
\lambda \cdot \mathrm{g}[\boldsymbol{\phi}] = -\log[Pr(\boldsymbol{\phi})]
$$


## L2 regularization

- Most common regularizer is **L2 regularization**
- Favors smaller parameters (like in previous example)

$$
\hat{\boldsymbol{\phi}} = \underset{\boldsymbol{\phi}}{\mathrm{argmin}}\left[\mathrm{L}[\boldsymbol{\phi}, \{\mathbf{x}_i, \mathbf{y}_i\}] + \frac{\lambda}{2} \sum_j \phi_j^2\right]
$$

- The $\frac{1}{2}$ makes `weight_decay` $= \lambda$. UDL and the figure two slides on write $\lambda' \sum_j \phi_j^2$, so $\lambda = 2\lambda'$.
- Also called **Tikhonov regularization, ridge regression**
- In neural networks, usually just for weights, and called **weight decay**
- Probabilistic reading: a zero-mean Gaussian prior $\phi_j \sim \mathcal{N}(0, \sigma^2_\phi)$ on the parameters,
  $-\log Pr(\boldsymbol{\phi}) = \frac{1}{2\sigma^2_\phi}\sum_j \phi_j^2 + \text{const}$

## Why does L2 regularization help?

![](figs/06-Generalization-and-Regularization/deep-network-weights-biases.png), Omega_1 in R^{2x4}, Omega_2 in R^{3x2}, Omega_3 in R^{2x3} label each layer." width="70%"}

- Outputs are a weighted linear combination of last layer activations. Smaller weights attenuate changes.

- Same for the pre-activations into the last layer, and so on… all the way back.
- Discourages fitting excessively to the training data (overfitting); encourages
  smoothness between datapoints.


## L2 regularization (simple net from the first half)

![L2 regularization with increasing $\lambda'$ (UDL's convention, penalty $\lambda'\sum_j\phi_j^2$, so `weight_decay` $= 2\lambda'$; the panel titles call it $\lambda$). Black: true function; orange: training data; cyan: fitted model.](figs/06-Generalization-and-Regularization/l2-regularization-lambda-fits.png)

## Weight decay in PyTorch, and AdamW

**L2 penalty = weight decay.** Adding $\frac{\lambda}{2}\|\boldsymbol{\phi}\|^2$ to
the loss (the zero-mean Gaussian prior of the MAP view above) adds
$\lambda\boldsymbol{\phi}$ to the gradient, so each SGD step first shrinks every
weight by $(1 - \alpha\lambda)$:

$$
\boldsymbol{\phi} \leftarrow \boldsymbol{\phi} - \alpha\left(\frac{\partial L}{\partial \boldsymbol{\phi}} + \lambda\boldsymbol{\phi}\right)
= (1 - \alpha\lambda)\,\boldsymbol{\phi} - \alpha\frac{\partial L}{\partial \boldsymbol{\phi}}
$$

`weight_decay` is $\lambda$ in this notation.

- AdamW decouples the decay from the adaptive step; derivation and the 2026 recipe are in lecture 04

```python
torch.optim.SGD(params, lr=0.1, weight_decay=1e-4)    # L2 folded into the gradient
torch.optim.Adam(params, lr=1e-3, weight_decay=1e-4)  # same, then divided by sqrt(v): barely decays
torch.optim.AdamW(params, lr=1e-3, weight_decay=0.1)  # decoupled: the LLM default
```

The L2 experiment two slides back, on the toy model from the first half, where
the penalty has a closed form. With 14 hidden units and 15 training points the
unregularized model can pass through every point. Adding
$\lambda_{\text{fit}} \sum_j \omega_j^2$ (the bias $\beta$ is not penalized)
changes the normal equations to
$(\mathbf{A}^\top\mathbf{A} + \lambda_{\text{fit}}\mathbf{P})\boldsymbol{\phi} =
\mathbf{A}^\top\mathbf{y}$, where $\mathbf{A}$ holds the hidden-unit activations
and $\mathbf{P}$ is the identity with its bias entry zeroed. As
$\lambda_{\text{fit}}$ grows the weights shrink, the training loss rises, and
the fit smooths out until it underfits.

The second cell checks that the two views agree: gradient descent with the decay
update above (shrink every weight by $(1 - \alpha\lambda)$, then step along the
data gradient) lands on the closed-form solution. The penalty
$\lambda_{\text{fit}}\sum_j\omega_j^2$ is
$\frac{\lambda}{2}\lVert\boldsymbol{\omega}\rVert^2$ with
$\lambda = 2\lambda_{\text{fit}}$, so that is the `weight_decay` to use.

In [ ]:
#| code-fold: true
#| fig-cap: "Closed-form L2-regularized fits of the 14-hidden-unit toy model to 15 noisy points, for six values of $\\lambda_{\\text{fit}}$. Black: true function; orange: training data; blue: fitted model."
#| fig-alt: "Six panels of the fitted model (blue) against the true function (black) and 15 training points (orange) for lambda = 0, 0.00001, 0.0001, 0.001, 0.01 and 0.1. At lambda = 0 the fit passes through every point and plunges below -1 between two points near x = 0.6. The dip shrinks as lambda grows; at 0.01 the fit is smooth and close to the true function, and at 0.1 it is too flat to reach the peak or the trough."
# Adapted from UDL notebook 9.1 (MIT)
D_l2 = 14
rng = np.random.default_rng(1)
x_l2, y_l2 = generate_data(15, rng)
A_l2 = features(x_l2, D_l2)
P = np.eye(D_l2 + 1)
P[0, 0] = 0                                    # do not penalize the bias

def ridge_fit(lam_fit):
    return np.linalg.solve(A_l2.T @ A_l2 + lam_fit * P, A_l2.T @ y_l2)

lambdas = [0, 1e-5, 1e-4, 1e-3, 1e-2, 1e-1]
fig, axes = plt.subplots(2, 3, figsize=(10, 5.2), sharex=True, sharey=True)
for ax, lam_fit in zip(axes.flat, lambdas):
    phi = ridge_fit(lam_fit)
    train_loss = np.sum((A_l2 @ phi - y_l2) ** 2)
    ax.plot(x_grid, mu, "k-")
    ax.plot(x_l2, y_l2, "o", color="#eb6834", mec="#52514e", ms=4)
    ax.plot(x_grid, features(x_grid, D_l2) @ phi, color="#2a78d6")
    ax.set_ylim(-1, 4)
    ax.set_title(rf"$\lambda$ = {lam_fit:g}:  $\|\omega\|$ = {np.linalg.norm(phi[1:]):.0f}, "
                 f"train loss = {train_loss:.2f}", fontsize=9)
for ax in axes[1]:
    ax.set_xlabel("$x$")
for ax in axes[:, 0]:
    ax.set_ylabel("$y$")
plt.tight_layout()
plt.show()

In [ ]:
lam_fit = 1e-2
weight_decay = 2 * lam_fit                   # lambda in the update rule
H = 2 * A_l2.T @ A_l2 + weight_decay * P     # Hessian of the regularized loss
alpha = 1.0 / np.linalg.eigvalsh(H).max()    # a safe step size for this quadratic
decay = np.ones(D_l2 + 1)
decay[1:] = 1 - alpha * weight_decay         # shrink the weights, not the bias

phi = np.zeros(D_l2 + 1)
for step in range(100_000):
    grad_data = 2 * A_l2.T @ (A_l2 @ phi - y_l2)     # dL/dphi for the sum of squares
    phi = decay * phi - alpha * grad_data
print(f"alpha = {alpha:.4f}, weights shrink by {decay[1]:.6f} per step")
print(f"largest |phi_GD - phi_closed_form| at lambda_fit = {lam_fit:g}: "
      f"{np.abs(phi - ridge_fit(lam_fit)).max():.1e}")

# Implicit regularization

Even without an explicit penalty, the optimization algorithm itself regularizes:
discrete gradient descent and SGD behave as if they were minimizing a modified
loss. This is the mechanism that answers the first half's question of why training
picks the smooth solution.


## Implicit regularization of gradient descent

$$
\phi_{t+1} = \phi_t - \alpha \frac{\partial L[\phi_t]}{\partial \phi}
\quad \xrightarrow{\lim_{\alpha \to 0}} \quad
\frac{d\phi}{dt} = -\frac{\partial L}{\partial \phi}
$$

- In the limit, as $\alpha \rightarrow 0$, the gradient descent equation becomes the
  **gradient flow** differential equation.
- Discrete gradient descent doesn't converge to the same place.

- The difference acts as an implicit regularization, which can be derived
  (see the notes at the end of UDL Ch. 9):

$$
\tilde{\mathrm{L}}_{GD}[\boldsymbol{\phi}] = \mathrm{L}[\boldsymbol{\phi}] + \underbrace{\frac{\alpha}{4}\left\lVert \frac{\partial L}{\partial \boldsymbol{\phi}} \right\rVert^2}_{\text{penalty for large gradients}}
$$


![](figs/06-Generalization-and-Regularization/gradient-flow-vs-descent.png)


## Implicit regularization: gradient flow vs. gradient descent

![](figs/06-Generalization-and-Regularization/implicit-regularization-gradient-flow.png)

**a)** Gradient descent doesn't converge to same location as (continuous) gradient flow.


**b)** Plot of the implicit regularization ($\sim \lVert \partial L / \partial \phi \rVert^2$) to be added to loss


**c)** With regularization, continuous descent converges to same place



## Implicit regularization of SGD

- Gradient descent disfavors areas where gradients are steep

$$
\tilde{\mathrm{L}}_{GD}[\boldsymbol{\phi}] = \mathrm{L}[\boldsymbol{\phi}] + \frac{\alpha}{4}\left\lVert \frac{\partial L}{\partial \boldsymbol{\phi}} \right\rVert^2
$$

- SGD likes all batches to have similar gradients

$$
\begin{aligned}
\tilde{\mathrm{L}}_{SGD}[\boldsymbol{\phi}] &= \tilde{\mathrm{L}}_{GD}[\boldsymbol{\phi}] + \underbrace{\frac{\alpha}{4B}\sum_{b=1}^{B}\left\lVert \frac{\partial L_b}{\partial \boldsymbol{\phi}} - \frac{\partial L}{\partial \boldsymbol{\phi}} \right\rVert^2} \\
&= \mathrm{L}[\boldsymbol{\phi}] + \frac{\alpha}{4}\left\lVert \frac{\partial L}{\partial \boldsymbol{\phi}} \right\rVert^2 + \frac{\alpha}{4B}\sum_{b=1}^{B}\left\lVert \frac{\partial L_b}{\partial \boldsymbol{\phi}} - \frac{\partial L}{\partial \boldsymbol{\phi}} \right\rVert^2
\end{aligned}
$$

Want the batch variance to be small, rather than some batches fitting well and others not well…

where
$L = \frac{1}{I}\sum_{i=1}^{I} \ell_i[\mathbf{x}_i, y_i]$ and
$L_b = \frac{1}{|\mathcal{B}|}\sum_{i \in \mathcal{B}_b} \ell_i[\mathbf{x}_i, y_i]$:
$B$ is the number of batches, $\mathcal{B}_b$ the set of examples in batch $b$, and $|\mathcal{B}|$ the batch size.

- Depends on <u>learning rate</u> – perhaps why larger learning rates generalize better.

See derivation at end of UDL Ch. 9.

## Loss and regularization surfaces

![Original Gabor model loss, the two implicit regularization terms, and the modified SGD loss.](figs/06-Generalization-and-Regularization/gd-sgd-modified-loss-surfaces.png)


**b)** GD modification:
$$
\frac{\alpha}{4}\left\lVert \frac{\partial L}{\partial \boldsymbol{\phi}} \right\rVert^2
$$

**c)** SGD modification:
$$
\frac{\alpha}{4B}\sum_{b=1}^{B}\left\lVert \frac{\partial L_b}{\partial \boldsymbol{\phi}} - \frac{\partial L}{\partial \boldsymbol{\phi}} \right\rVert^2
$$

**d)** Modified loss:
$$
\begin{aligned}
&\tilde{\mathrm{L}}_{SGD}[\boldsymbol{\phi}] \\
&= \mathrm{L}[\boldsymbol{\phi}] + \frac{\alpha}{4}\left\lVert \frac{\partial L}{\partial \boldsymbol{\phi}} \right\rVert^2 \\
&\quad + \frac{\alpha}{4B}\sum_{b=1}^{B}\left\lVert \frac{\partial L_b}{\partial \boldsymbol{\phi}} - \frac{\partial L}{\partial \boldsymbol{\phi}} \right\rVert^2
\end{aligned}
$$


## Learning rate and batch size on MNIST-1D

![UDL Fig. 9.5, a capacity sweep: each point is a separately trained two-hidden-layer network of that width, trained for 6000/LR iterations.](figs/06-Generalization-and-Regularization/mnist1d-learning-rate-batch-size.png)

Generally, performance is

- best for larger learning rates
- best with smaller batches


**Recap**: implicit regularization of GD and SGD

- Larger learning rates may lead to better generalization
- SGD seems to favor places where gradients are stable (all batches agree on slope)
- SGD generalizes better than GD, and smaller batches generally beat larger ones


# Early stopping

A simple way to stop the weights from overfitting is not to let training run to
convergence.


## Early stopping

![Simplified shallow network model with 14 linear regions initialized randomly (cyan curve in (a)) and trained with SGD using a batch size of five and a learning rate of 0.05.](figs/06-Generalization-and-Regularization/early-stopping-iterations.png)

- If we stop training early, weights don't have time to overfit to noise; they start
  small and don't have time to get large, which reduces effective model complexity.
- Don't have to re-train with different hyper-parameters – just "checkpoint"
  regularly and pick the model with lowest validation loss.


# Ensembling

Instead of relying on one model, we can train several and combine their
predictions.


## Ensembling

- Combine several models – an **ensemble**
- Combining outputs

|                | Mean                | Median/Frequent (Robust)       |
|----------------|---------------------|--------------------------------|
| Regression     | Mean of outputs     | Median of outputs              |
| Classification | Mean before softmax | Most frequent predicted class  |

- Can be simply different initializations or even different models
- Or train with different subsets of the data resampled with replacement –
  **bootstrap aggregating (bagging)**


![Ensembling via bagging. Size of orange point indicates number of times the data point was re-sampled.](figs/06-Generalization-and-Regularization/bagging-ensemble.png)



# Dropout

Dropout injects randomness into the network itself during training, which also
connects back to ensembling.


## Dropout

Randomly clamp ~50% of hidden units to 0 on each iteration.

![](figs/06-Generalization-and-Regularization/dropout-iterations.png)

- Makes the network less dependent on any given hidden unit.
- At test time, all hidden units are active, which was not the case during training
  - Must rescale using the *weight scaling inference rule* – multiply weights by (1 – dropout probability)
- PyTorch's `nn.Dropout` is *inverted dropout*: it scales by $1/(1-p)$ during training and does nothing at test time.

Why the scaling matters, in a few lines. Each training step draws a fresh mask
that keeps a unit with probability $1 - p$. Averaged over masks, a dropped-out
activation is $(1-p)\,h$, smaller than the $h$ the next layer sees at test
time. Either rescale at test time (the weight scaling inference rule) or, as
PyTorch does, divide by $1 - p$ during training so the expected activation
already equals $h$. Change `p` and check that the last two rows still agree.

In [ ]:
p = 0.5                                          # dropout probability
h = np.array([0.8, 1.5, 0.0, 2.3, 0.4, 1.1])     # activations of one hidden layer
rng = np.random.default_rng(0)

mask = rng.uniform(size=(100_000, h.size)) >= p  # one row per training step
print("one mask:                ", mask[0].astype(int))
print("activations h:           ", h)
print("E[mask * h]:             ", np.round((mask * h).mean(axis=0), 3), " ~ (1 - p) h")
print("E[mask * h / (1 - p)]:   ", np.round((mask * h / (1 - p)).mean(axis=0), 3), " ~ h  (inverted dropout)")
print("fraction of units kept:  ", round(mask.mean(), 3))

## Dropout removes kinks, and Monte Carlo dropout

![](figs/06-Generalization-and-Regularization/dropout-removes-kinks.png)

- Prevents situations where subsequent hidden units correct for excessive swings from earlier hidden units
- Can eliminate kinks in the function that are far from data and don't contribute to training loss

**Monte Carlo dropout for inference**: run the network multiple times with
different random subsets of units clamped to zero (as in training) and combine the
results with an ensembling method. Every random version of the network is a
different model, but we do not have to train or store multiple networks.


# Adding noise

Dropout is one way of injecting randomness during training. Noise can also be
added to the inputs, the weights or the labels, each with its own regularizing
effect.


## Adding noise

![Adding noise to input with different variances.](figs/06-Generalization-and-Regularization/noise-on-inputs.png)

- to inputs – induces weight regularization (see Exercise 9.3 in UDL)
- to weights – makes robust to small weight perturbations
- to outputs (labels) – reduces "overconfident" probability for target class (label smoothing)

# Transfer learning, multi-task learning, self-supervised learning

The remaining methods work by bringing in more information, either from other
tasks or from more (or modified) data. Strictly speaking they are not
regularization, but they improve generalization when dataset sizes are limited.


## Transfer learning

![](figs/06-Generalization-and-Regularization/transfer-learning-segmentation-depth.png)

Strictly speaking not regularization, but can help improve generalization when
dataset sizes are limited.

(1) Train the model for segmentation
  - Assume we have lots of segmentation training data

(2) Replace the final layers to match the new task
  - Assume we have limited depth training data


(3) Either:

**a)** Freeze the rest of the layers and train the final layers

**b)** Fine tune the entire model


## Multi-task learning

![](figs/06-Generalization-and-Regularization/multitask-learning-segmentation-depth.png)

- Train the model for 2 or more tasks simultaneously
  - Weighted combo of loss functions

$$
L_{total} = \alpha \cdot L_{segmentation} + \beta \cdot L_{depth}
$$

- Less likely to overfit to training data of one task
- Can be harder to get training to converge. Might have to vary the individual task loss weightings, $\alpha$ and $\beta$.

## Self-supervised learning

![](figs/06-Generalization-and-Regularization/self-supervised-inpainting.png)

*The animal didn't cross the* \_\_\_\_\_\_ *because it was too tired.*

- Mask out part of the training data
- Train model to try to infer missing data
  - masked data is the target
- $\Rightarrow$ Model learns characteristics of the data
- Then apply transfer learning

# Data augmentation

If more data reduces variance and we cannot collect more, we can manufacture
plausible variations of the data we have.


## Data augmentation

![](figs/06-Generalization-and-Regularization/image-augmentation-examples.png)

Transform each training example in ways that do not change its label, so the
model sees many plausible variants of the data you have.


```python
import torch
import torchvision.transforms as transforms

# Define augmentation pipeline
transform = transforms.Compose([
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.3),
    transforms.RandomRotation(degrees=30),
    transforms.ColorJitter(brightness=0.5, contrast=0.5, saturation=0.5),
    transforms.RandomAffine(degrees=20, translate=(0.2, 0.2), shear=10),
    transforms.RandomPerspective(distortion_scale=0.5, p=0.5),
    transforms.ToTensor(),  # Convert image to tensor
])

# Apply transformations multiple times to visualize augmentation
augmented_image = transform(image)
```

<https://pytorch.org/vision/main/transforms.html>


## Data augmentation: MNIST-1D

![](figs/06-Generalization-and-Regularization/mnist1d-augmentation-error.png)


Examples in training set: 4000\
Examples in test set: 1000\
Length of each example: 40

- Randomly circularly rotate
- randomly scale between 0.8 and 1.2


# Summary

The two halves of this lecture are one story: test error decomposes into noise,
bias and variance; big networks escape the classical trade-off because something
makes them interpolate smoothly; and regularization, explicit or implicit, is that
something.


## Regularization overview

![](figs/06-Generalization-and-Regularization/regularization-overview.png)

## Summary

- MNIST-1D is small enough to experiment on and hard enough to separate logistic
  regression, MLPs, CNNs and RNNs; training error goes to zero while test error does not.
- For least-squares regression, expected test error decomposes exactly into
  **variance** (over training sets) + **bias** (of the average model) + **noise** $\sigma^2$.
- More data reduces variance; more capacity reduces bias but raises variance: the
  classical bias–variance trade-off.
- Past the interpolation threshold (parameters $\approx$ examples) test error falls
  again: **double descent**. Big networks interpolate smoothly between data points.
- In high dimensions almost all the volume is between the data points (curse of
  dimensionality), so how the model interpolates is what generalization measures.
- Pick capacity and other hyperparameters on a **validation set**; touch the test set once.
- **Explicit regularization** adds $\lambda\,\mathrm{g}[\boldsymbol{\phi}]$ to the loss;
  it is a prior (MAP). L2 = weight decay; with Adam use the decoupled **AdamW** form.
- **Implicit regularization**: finite-step GD minimizes
  $L + \frac{\alpha}{4}\lVert\partial L/\partial\boldsymbol{\phi}\rVert^2$, and SGD
  adds a batch-variance penalty; larger learning rates and smaller batches generalize better.
- **Early stopping**, **ensembles** (bagging), **dropout** (and Monte Carlo dropout)
  and **added noise** each reduce variance or smooth the function.
- **Transfer**, **multi-task** and **self-supervised** learning bring in data from
  other tasks; **data augmentation** manufactures more from the data you have.